In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import logging
import datetime as dt
import pandas as pd
# main.py
from pathlib import Path
from utils.fx import  update_daily_rates, fetch_twbank_exchange, update_file
# from utils.news import update_cnn_news
from utils import plot
from utils.config import CURRENCIES

LOG_DIR = Path("log")
LOG_DIR.mkdir(exist_ok=True)

log_file = LOG_DIR / f"run_{dt.datetime.now():%Y%m%d_%H%M%S}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info("Program started")

2026-03-04 10:33:49,082 | INFO     | __main__ | Program started


In [4]:
today = dt.datetime.today()

###################################################################
# Step 1. Download Foreign Exchange Rate                          #
###################################################################
# Historical Foreign Exchange Rate - Closing Rate
now_utc = dt.datetime.now(dt.timezone.utc)
if now_utc.hour == 11: # Execute one time everyday
    output1 = fetch_twbank_exchange(URL="https://rate.bot.com.tw/xrt/all/day",
                                    currencies=CURRENCIES)
    # Append data to dataset
    update_file(dataframe = output1,
                update_file="Data/history/Historical_{currency}.csv",
                currencies= CURRENCIES)

# Foreign Exchange Rate - Non-Business Hours
output2 = fetch_twbank_exchange(URL="https://rate.bot.com.tw/xrt?Lang=en-US",
                                save_html=True,
                                save_directory=f"Data/Temporary Save/ExchangeRate@{today:%Y%m%d%H%M%S}.csv")
# Append data to dataset
update_file(dataframe = output2,
            update_file="Data/Foreign Exchage Rate/Foreign Exchage Rate_{currency}.csv",
            currencies= CURRENCIES)

In [5]:
output2

,Date,Currency,Cash,Cash.1,Spot,Spot.1
0,2026-03-04 17:21:00,USD,31.29500,31.96500,31.6200,31.7700
1,2026-03-04 17:21:00,HKD,3.89900,4.10300,4.0200,4.0900
2,2026-03-04 17:21:00,GBP,41.17000,43.29000,42.0650,42.6950
3,2026-03-04 17:21:00,AUD,21.89000,22.67000,22.1050,22.4500
4,2026-03-04 17:21:00,CAD,22.68000,23.59000,23.0100,23.3400
5,2026-03-04 17:21:00,SGD,24.24000,25.15000,24.7100,24.9300
6,2026-03-04 17:21:00,CHF,39.71000,40.91000,40.3200,40.7100
7,2026-03-04 17:21:00,JPY,0.19230,0.20510,0.1991,0.2041
8,2026-03-04 17:21:00,ZAR,NaN,NaN,1.8810,1.9710
9,2026-03-04 17:21:00,SEK,NaN,NaN,3.3800,3.5000


In [ ]:
###################################################################
# Step 2. Generate Plotly figure                                  #
###################################################################
for currency in CURRENCIES:
    csv_file = f'Data/history/Historical_{currency}.csv'
    plot.plot_history(csv_file_path=csv_file, 
                        currency=currency,
                        show_html = True,
                        save_html = False,
                        save_directory = f"assets/plot_history_{currency}.html")
    csv_file = f'Data/Foreign Exchage Rate/Foreign Exchage Rate_{currency}.csv'
    plot.plot_now(csv_file_path =csv_file,
                    currency = currency,
                    show_html = True,
                    save_html = False,
                    save_directory = f"assets/plot_now_{currency}.html")
    break

ValueError: unconverted data remains when parsing with format "%Y-%m-%d %H:%M": ":00", at position 631. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.